This will simulate an event bridge that runs periodically (hourly, daily, etc.) to identify new markets that match. 

It will fetch data from Kalshi and Polymarket, check the database, and append new events along with any new matches to our `arb_candidates` database.

In [27]:
import pandas as pd
import requests
import os
import json

from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz
import hashlib

from kalshi_client import KalshiEventClient
from polymarket_client import PolymarketEventClient
from utils import create_match_keys, clean_text_cols

In [28]:
MODEL = SentenceTransformer('all-MiniLM-L6-v2')
EVENT_LIMIT = 10_000

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5561.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
## --- 1. Generic Matching Engine ---

def get_matches(df_k, df_p, text_col, id_col, type_label='event', threshold=0.65, strict_key_match=True):
    """
    Flexible Matcher with Mode Selection:
    
    strict_key_match=True:  Only matches within the same 'match_key' (Fast, strict).
        Useful for markets where the 'match_key' is reliable and you want to maximize performance,
        like event-level matching where the 'match_key' is often a strong signal.
        
    strict_key_match=False: Compares every Kalshi row to every Poly row (Slow, discovery mode).
        Useful for markets where the 'match_key' may not be reliable or for a final comprehensive pass, 
        and with small datasets (like markets) where performance is less of a concern.
    """
    all_results = []

    if strict_key_match:
        # MODE A: Optimized Blocking (The Fast Path)
        common_keys = set(df_k['match_key']) & set(df_p['match_key'])
        blocks = [(key, key) for key in common_keys]
        print(f"Blocking Mode: Processing {len(blocks)} shared keys...")
    else:
        # MODE B: Global Search (The Discovery Path)
        # We create one giant "virtual block" containing all rows
        blocks = [('GLOBAL', 'GLOBAL')]
        print("Global Mode: Comparing all rows (Cross-Join)...")

    for k_key, p_key in blocks:
        # Filter subsets based on mode
        if strict_key_match:
            k_subset = df_k[df_k['match_key'] == k_key].reset_index(drop=True)
            p_subset = df_p[df_p['match_key'] == p_key].reset_index(drop=True)
        else:
            k_subset = df_k.reset_index(drop=True)
            p_subset = df_p.reset_index(drop=True)
        
        if k_subset.empty or p_subset.empty:
            continue

        # 1. Vectorize
        k_titles = k_subset[text_col].astype(str).tolist()
        p_titles = p_subset[text_col].astype(str).tolist()
        
        k_embeddings = MODEL.encode(k_titles, convert_to_tensor=True)
        p_embeddings = MODEL.encode(p_titles, convert_to_tensor=True)
        cosine_scores = util.cos_sim(k_embeddings, p_embeddings)

        # 2. Extract Matches
        for i, k_text in enumerate(k_titles):
            best_idx = cosine_scores[i].argmax().item()
            semantic_score = cosine_scores[i][best_idx].item()
            
            if semantic_score >= threshold:
                p_text = p_titles[best_idx]
                key_sim = fuzz.token_sort_ratio(k_text, p_text) / 100.0
                
                all_results.append({
                    f'{type_label}_ids': {
                        'kalshi': k_subset.iloc[i][id_col],
                        'poly': p_subset.iloc[best_idx][id_col]
                    },
                    'kalshi_text': k_text,
                    'poly_text': p_text,
                    'semantic_score': round(semantic_score, 4),
                    'key_sim_score': round(key_sim, 4),
                    'total_score': round((semantic_score + key_sim) / 2, 4),
                    'match_key': {
                        'kalshi': k_subset.iloc[i]['match_key'],
                        'poly': p_subset.iloc[best_idx]['match_key']
                    }
                })

    if not all_results:
        return pd.DataFrame()
        
    return pd.DataFrame(all_results).sort_values(by='total_score', ascending=False).reset_index(drop=True)

## --- 2. Data Hydration & Cleaning ---
def hydrate_and_clean_markets(match_df, event_id_col='event_ids'):
    """Fetches raw market data and applies the cleaning function immediately."""
    ids_list = match_df[event_id_col].tolist()
    poly_ids = [d.get('poly') for d in ids_list if d.get('poly')]
    kalshi_tickers = [d.get('kalshi') for d in ids_list if d.get('kalshi')]

    poly_rows, kalshi_rows = [], []

    # Polymarket Fetch
    if poly_ids:
        try:
            url = "https://gamma-api.polymarket.com/events"
            response = requests.get(url, params=[('id', eid) for eid in poly_ids], timeout=10)
            if response.status_code == 200:
                for event in response.json():
                    for m in event.get('markets', []):
                        # Construct descriptive text for matching
                        outcomes = str(m.get('outcomes', '')).replace("[", "").replace("]", "").replace("'", "")
                        poly_rows.append({
                            'event_id': str(event.get('id')),
                            'market_id': m.get('id'),
                            # 'descriptive_text': f"{m.get('question', '')} {outcomes}"
                            'descriptive_text': f"{m.get('question', '')}"
                            
                        })
        except Exception as e: print(f"Poly Hydration Error: {e}")

    # Kalshi Fetch
    for ticker in kalshi_tickers:
        try:
            url = f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}?with_nested_markets=true"
            res = requests.get(url, timeout=5)
            if res.status_code == 200:
                event_data = res.json().get('event', {})
                for m in event_data.get('markets', []):
                    kalshi_rows.append({
                        'event_id': ticker,
                        'market_id': m.get('ticker'),
                        'descriptive_text': m.get('title', '')
                    })
        except Exception as e: print(f"Kalshi Hydration Error: {e}")

    # Convert to DataFrames and apply your custom cleaning
    df_p = pd.DataFrame(poly_rows)
    df_k = pd.DataFrame(kalshi_rows)

    if not df_p.empty:
        df_p = clean_text_cols(df_p, exclude_cols=['market_id', 'event_id'])
        df_p = create_match_keys(df_p, ['descriptive_text'])
    
    if not df_k.empty:
        df_k = clean_text_cols(df_k, exclude_cols=['market_id', 'event_id'])
        df_k = create_match_keys(df_k, ['descriptive_text'])

    return df_k, df_p


In [30]:
import pandas as pd
import os

MASTER_FILE = 'data/events/master_events.json'

# Ensure the directory exists
os.makedirs(os.path.dirname(MASTER_FILE), exist_ok=True)

kalshi_client = KalshiEventClient(event_limit=EVENT_LIMIT)
poly_client = PolymarketEventClient(event_limit=EVENT_LIMIT)


print("Fetching new data...")
kalshi_client.fetch_events()
poly_client.fetch_events()

new_k_events = kalshi_client.transform_events()
new_p_events = poly_client.transform_events()

# Combine all newly fetched data
df_events_new = pd.concat([new_k_events, new_p_events], ignore_index=True)
print('Adding match keys to new data...')
df_events_new = create_match_keys(df_events_new, ['event_title']) # add new keys to new data

if os.path.exists(MASTER_FILE) and os.path.getsize(MASTER_FILE) > 0:
    df_events_old = pd.read_json(MASTER_FILE, lines=True)
    # Combine old and new
    print("Master file found. Combining with new data...")
    df_master = pd.concat([df_events_old, df_events_new], ignore_index=True)
else:
    print("Master file not found or empty. Creating new record.")
    df_master = df_events_new

if not df_master.empty:
    subset_cols = ['event_id', 'platform'] if 'event_id' in df_master.columns else None
    df_master = df_master.drop_duplicates(subset=subset_cols, keep='last')

# Save back to JSON (Lines=True is generally better for JSON datasets)
df_master.to_json(MASTER_FILE, orient='records', lines=True)
print(f"Sync complete. Total records in master: {len(df_master)}")

Fetching new data...
Fetching up to 10000 open events from Kalshi...
Fetching up to 10000 Polymarket events...
Adding match keys to new data...
Master file found. Combining with new data...
Sync complete. Total records in master: 14713


In [34]:
k_events = df_master[df_master['platform'] == 'kalshi']
p_events = df_master[df_master['platform'] == 'poly']

print("Matching events...")
matched_events = get_matches(
    k_events,
    p_events,
    'event_title',
    'event_id',
    type_label='event', 
    threshold=0.70,
    strict_key_match=True
)

matched_events.head()

Matching events...
Blocking Mode: Processing 94 shared keys...


,event_ids,kalshi_text,poly_text,semantic_score,key_sim_score,total_score,match_key
0,"{'kalshi': 'KXFACUP-26', 'poly': '179914'}",fa cup winner 2025 2026,2025 2026 fa cup winner,0.9843,1.00,0.9921,"{'kalshi': '2026_2025', 'poly': '2026_2025'}"
1,"{'kalshi': 'KXUPONLY-01', 'poly': '127122'}",which guests will appear on the uponly podcast...,which guests will appear on the uponly podcast...,0.9955,0.94,0.9677,"{'kalshi': 'unknown_2027', 'poly': 'unknown_20..."
2,"{'kalshi': 'KXBTCVSGOLD-26', 'poly': '133707'}",bitcoin outperform gold in 2026 in 2026,bitcoin outperform gold in 2026,0.9920,0.89,0.9410,"{'kalshi': '2026_bitcoin_2026', 'poly': '2026_..."
3,"{'kalshi': 'KXGREENIND-27', 'poly': '160520'}",greenland vote for independence in 2026 before...,greenland vote for independence in 2026,0.9837,0.87,0.9268,"{'kalshi': '2026_independence_2026', 'poly': '..."
4,"{'kalshi': 'KXMLBAL-26', 'poly': '215866'}",american league champion 2026,mlb 2026 american league champion,0.9132,0.94,0.9266,"{'kalshi': '2026_american_league_2026', 'poly'..."


In [ ]:
print("Hydrating and cleaning markets for matched events...")
k_markets_clean, p_markets_clean = hydrate_and_clean_markets(matched_events)

print("Matching markets...")
matched_markets = get_matches(
    k_markets_clean, 
    p_markets_clean, 
    'descriptive_text', 
    'market_id', 
    type_label='market', 
    threshold=0.70,
    strict_key_match=False
)

matched_markets

Matching markets...
Global Mode: Comparing all rows (Cross-Join)...


,market_ids,kalshi_text,poly_text,semantic_score,key_sim_score,total_score,match_key
0,"{'kalshi': 'KXBTCVSGOLD-26', 'poly': '1068346'}",bitcoin outperform gold in 2026,bitcoin outperform gold in 2026,1.0000,1.00,1.0000,"{'kalshi': '2026_bitcoin_2026', 'poly': '2026_..."
1,"{'kalshi': 'KXALBUMRELEASE-26-OLIV', 'poly': '...",olivia rodrigo release a new album in 2026,olivia rodrigo release an album in 2026,0.9897,0.91,0.9498,"{'kalshi': '2026_new_rodrigo_2026', 'poly': '2..."
2,"{'kalshi': 'KXALBUMRELEASE-26-TAY', 'poly': '1...",taylor swift release a new album in 2026,taylor swift release an album in 2026,0.9892,0.91,0.9496,"{'kalshi': '2026_new_taylor_2026', 'poly': '20..."
3,"{'kalshi': 'KXALBUMRELEASE-26-TRA', 'poly': '1...",travis scott release a new album in 2026,travis scott release an album in 2026,0.9866,0.91,0.9483,"{'kalshi': '2026_new_scott_2026', 'poly': '202..."
4,"{'kalshi': 'KXALBUMRELEASE-26-SAB', 'poly': '1...",sabrina carpenter release a new album in 2026,sabrina carpenter release an album in 2026,0.9906,0.90,0.9453,"{'kalshi': '2026_new_2026', 'poly': '2026_2026'}"
...,...,...,...,...,...,...,...
265,"{'kalshi': 'KXALBUMRELEASE-26-WEE', 'poly': '9...",weeknd release a new album in 2026,weeknd be the top spotify artist for 2026,0.7377,0.53,0.6338,"{'kalshi': '2026_new_2026', 'poly': '2026_top_..."
266,"{'kalshi': 'KXSONGRELEASE-27JAN01-JCO', 'poly'...",j cole release a new song 2026,kendrick lamar release an album in 2026,0.7152,0.55,0.6326,"{'kalshi': '2026_new_2026', 'poly': '2026_2026'}"
267,"{'kalshi': 'KXSONGRELEASE-27JAN01-KHA', 'poly'...",khalid release a new song 2026,kendrick lamar release an album in 2026,0.7138,0.55,0.6319,"{'kalshi': '2026_new_2026', 'poly': '2026_2026'}"
268,"{'kalshi': 'KXNFLDRAFTQB-26P2-NONE', 'poly': '...",no player be the 2nd quarterback drafted,1st pick in the 2026 pro football draft be a qb,0.7167,0.51,0.6134,"{'kalshi': 'unknown_2026', 'poly': '2026_footb..."


In [36]:
ARB_FILE_PATH = 'data/arb_candidates.json'
THRESHOLD_FOR_ARB_CANDIDATES = 0.90

new_matches = matched_markets[matched_markets['total_score'] > THRESHOLD_FOR_ARB_CANDIDATES].copy()

def create_arb_id(row):
    m_ids = row.get('market_ids', {})
    k_id = str(m_ids.get('kalshi', ''))
    p_id = str(m_ids.get('poly', ''))    
    combined = "".join(sorted([k_id, p_id]))
    return hashlib.md5(combined.encode()).hexdigest()[:10]


if not new_matches.empty:
    # Generate match_id for the new batch
    new_matches['match_id'] = new_matches.apply(create_arb_id, axis=1)

    if os.path.exists(ARB_FILE_PATH):
        existing_df = pd.read_json(ARB_FILE_PATH)
        # Combine existing and new
        combined_df = pd.concat([existing_df, new_matches], ignore_index=True)
        print("Comparing new matches against existing records...")
    else:
        combined_df = new_matches
        print("Creating new arb_candidates file...")
        
    before_count = len(combined_df)
    combined_df = combined_df.drop_duplicates(subset=['match_id']).reset_index(drop=True)
    after_count = len(combined_df)
    
    combined_df.to_json(ARB_FILE_PATH, orient='records', indent=4)
    print(f"Success: Updated {ARB_FILE_PATH}.")
    print(f"Total records: {after_count} (Added {after_count - (before_count - len(new_matches))} new unique pairs)")
else:
    print("No matches above 0.90 found. Skipping update.")
    combined_df = pd.read_json(ARB_FILE_PATH) if os.path.exists(ARB_FILE_PATH) else pd.DataFrame()

combined_df

Comparing new matches against existing records...
Success: Updated data/arb_candidates.json.
Total records: 131 (Added 0 new unique pairs)


,market_ids,kalshi_text,poly_text,semantic_score,key_sim_score,total_score,match_key,match_id
0,"{'kalshi': 'KXBTCVSGOLD-26', 'poly': '1068346'}",bitcoin outperform gold in 2026,bitcoin outperform gold in 2026,1.0000,1.00,1.0000,"{'kalshi': '2026_bitcoin_2026', 'poly': '2026_...",037fcde160
1,"{'kalshi': 'KXALBUMRELEASE-26-OLIV', 'poly': '...",olivia rodrigo release a new album in 2026,olivia rodrigo release an album in 2026,0.9897,0.91,0.9498,"{'kalshi': '2026_new_rodrigo_2026', 'poly': '2...",46f82f47c0
2,"{'kalshi': 'KXALBUMRELEASE-26-TAY', 'poly': '1...",taylor swift release a new album in 2026,taylor swift release an album in 2026,0.9892,0.91,0.9496,"{'kalshi': '2026_new_taylor_2026', 'poly': '20...",58fb9fa0ea
3,"{'kalshi': 'KXALBUMRELEASE-26-TRA', 'poly': '1...",travis scott release a new album in 2026,travis scott release an album in 2026,0.9866,0.91,0.9483,"{'kalshi': '2026_new_scott_2026', 'poly': '202...",e61b96ab61
4,"{'kalshi': 'KXALBUMRELEASE-26-SAB', 'poly': '1...",sabrina carpenter release a new album in 2026,sabrina carpenter release an album in 2026,0.9906,0.90,0.9453,"{'kalshi': '2026_new_2026', 'poly': '2026_2026'}",259c8ab8b2
...,...,...,...,...,...,...,...,...
126,"{'kalshi': 'KXPGATOUR-VATO26-JKEE', 'poly': '1...",john keefer win the valero texas open,johnny keefer win the 2026 valero texas open,0.9167,0.91,0.9134,"{'kalshi': 'john_texas_9999', 'poly': '2026_jo...",f63e78431a
127,"{'kalshi': 'KXPGATOUR-VATO26-MMCC', 'poly': '1...",matthew mccarty win the valero texas open,matt mccarty win the 2026 valero texas open,0.9235,0.90,0.9117,"{'kalshi': 'matthew_texas_9999', 'poly': '2026...",27d37bff25
128,"{'kalshi': 'KXPGATOUR-VATO26-JPOS', 'poly': '1...",j t poston win the valero texas open,jt poston win the 2026 valero texas open,0.9277,0.89,0.9089,"{'kalshi': 'texas_9999', 'poly': '2026_texas_2...",b8ea09a405
129,"{'kalshi': 'KXFOMEN-26-STR', 'poly': '1087533'}",january lennard struff win the french open,january lennard struff win the 2026 men s fren...,0.9259,0.88,0.9029,"{'kalshi': 'french_january_2026', 'poly': '202...",e7aad3c979
